# 9a. Stratosphere caches: build and check

Builds every cache the 9b-9i notebooks read, so they can run with `derivation_mode = "require"`, and
checks them: what exists, its size and coverage, and the one correction applied to the model data.

* **ERA5**: `ERA5/1x1/daily_zonal` (daily zonal means of U, V, T, Z and eddy fluxes on 37 levels; built
  by `ERA5/scripts/daily_zonal`) and its reduced indices (cached here); ERA5 00Z 10-hPa maps of the
  observed 2025/26 winter (for `WINTER = 2025` in 9h).
* **Model**: daily zonal means from the native `eam.h1` files, monthly zonal means of the post-processed
  U/T/Z3, daily 10-hPa height maps (native `Z010` remapped), for every `POOL` case and 2026 start.
* **Reanalysis climatologies**: monthly zonal means and the `Z10`, `PSL`, `TS` lead-time climatologies of
  every start month (May/Nov 1981-2010; Feb/Aug the six 1985-2010 starts: `CLIM_METHOD = "direct"`).

First build: about 1 h on a login node (most of it the daily model reads); later runs only check.

**The 9a-9i series** (polar vortex and SSWs in the RTENSO2026 2026 starts): `9a` builds and checks the
caches; `9b` model climatology; `9c` vortex forecast; `9d` SSW occurrence; `9e` ENSO/QBO background;
`9f` SSW dynamics; `9g` SSW mechanism (featured member vs controls); `9h` SSW-event animation; `9i`
seasonal-evolution animation. Shared loading: `util/strat_diag.py`; readers `util/strat_case.py`; maps
`util/vortex_plot.py`.

In [1]:
%matplotlib inline

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from IPython.display import Image, display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from util import rtenso_case as rc
from util import strat_case as sc
from util import strat_diag as sd

warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice")

### Run controls

In [2]:
NOTEBOOK = "9a_strat_build_caches"   # outputs: rc.OUTPUT_ROOT/9a_strat_build_caches/{figures,data}/
# "require": read the caches built by 9a only; "auto": build a missing cache; "rebuild": recompute
derivation_mode = "auto"
CLIM_METHOD = "direct"      # model climatology: "direct" (each start's Reanalysis starts; Feb/Aug 1985-2010), "lead_drift"
WORKERS = 4                 # processes reading model output (cache builds only; login node: 30 GB per user)

# Data locations (machine-specific)
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")                       # reference-data archive
ERA5_ZONAL_DIR = OBSERVATION_ROOT / "ERA5" / "1x1" / "daily_zonal"                    # ERA5 daily zonal means
ERA5_DAILY_DIR = OBSERVATION_ROOT / "ERA5" / "1x1" / "daily"                          # ERA5 daily maps (PSL)
SIMULATION_ROOT = Path("/global/cfs/cdirs/e3sm/S2S2D/RTENSO2026/simulation")         # raw eam.h1 history

cfg = sd.StratConfig(era5_zonal_dir=ERA5_ZONAL_DIR, era5_daily_dir=ERA5_DAILY_DIR, simulation_root=SIMULATION_ROOT,
                     clim_method=CLIM_METHOD, mode=derivation_mode, workers=WORKERS)
YEAR, INIT_MONTHS, POOL, CLIM_YEARS = cfg.year, cfg.init_months, cfg.pool, cfg.clim_years
ERA5_FIRST_YEAR, REF_LEVEL, SSW = cfg.era5_first_year, cfg.ref_level, cfg.ssw
MONTH_NAMES, INIT_COLORS, MODEL_PREFIX = sd.MONTH_NAMES, sd.INIT_COLORS, rc.MODEL_PREFIX
MAP_FILE = Path("/global/cfs/cdirs/e3sm/diagnostics/maps/map_ne30pg2_to_cmip6_180x360_aave.20200201.nc")
ERA5_MIRROR = Path("/global/cfs/projectdirs/m3522/cmip6/ERA5")           # hourly 0.25-degree ERA5 (10-hPa maps)
OBSERVED_WINTER = ("2025-10-01", "2026-06-30")                          # ERA5 10-hPa maps of the observed 2025/26 winter
CLIM_FIELDS = ("Z10", "PSL", "TS")                                      # Reanalysis map climatologies (9c, 9e, 9f, 9g-9i)

fontz = 13
fig_type = "png"
fig_dpi = 200
plt.rcParams.update({"font.size": fontz, "axes.titlesize": fontz, "axes.labelsize": fontz,
                     "legend.fontsize": fontz * 0.8, "xtick.labelsize": fontz * 0.85, "ytick.labelsize": fontz * 0.85})
rng = np.random.default_rng(0)


def save(fig, *parts):
    path = rc.save_figure(fig, NOTEBOOK, *parts, dpi=fig_dpi, ext=fig_type)
    print(f"[FIGURE] {path}")


DATA_DIR = rc.output_dir(NOTEBOOK, "data")

### 1. ERA5

The daily zonal-mean archive (years present, last day) and the reduced indices used by every notebook.

In [3]:
%%time
files = sorted(ERA5_ZONAL_DIR.glob("ERA5.daily_zonal.en00.U.*.nc"))
years = [int(f.name.split(".")[-2][:4]) for f in files]
missing = sorted(set(range(ERA5_FIRST_YEAR, max(years) + 1)) - set(years))
print(f"ERA5 daily zonal: {len(files)} years {min(years)}-{max(years)}" + (f"; missing {missing}" if missing else ""))
E = sd.load_era5(cfg)
print(f"last day {E.obs_end:%d %b %Y}; indices cached")
era5_maps = sc.era5_daily_maps(ERA5_MIRROR, *OBSERVED_WINTER, mode=derivation_mode, workers=WORKERS)
print(f"ERA5 10-hPa maps {pd.Timestamp(era5_maps['time'].values[0]):%d %b %Y} - {pd.Timestamp(era5_maps['time'].values[-1]):%d %b %Y}")
rc.obs_monthly_climatology("Z10", mode=derivation_mode)

ERA5 daily zonal: 48 years 1979-2026


ERA5 daily zonal means 01 Jan 1979 - 30 Jun 2026
last day 30 Jun 2026; indices cached


ERA5 10-hPa maps 01 Oct 2025 - 30 Jun 2026


CPU times: user 3.03 s, sys: 1.02 s, total: 4.06 s
Wall time: 34.8 s


<xarray.DataArray 'Z10' (month: 12, lat: 180, lon: 360)> Size: 3MB
array([[[31875.76 , 31875.748, 31875.744, ..., 31875.744, 31875.752,
         31875.756],
        [31874.254, 31874.258, 31874.252, ..., 31874.238, 31874.232,
         31874.258],
        [31872.26 , 31872.256, 31872.285, ..., 31872.275, 31872.268,
         31872.254],
        ...,
        [28718.426, 28717.883, 28717.342, ..., 28720.32 , 28719.65 ,
         28719.03 ],
        [28759.344, 28758.977, 28758.637, ..., 28760.555, 28760.123,
         28759.707],
        [28800.705, 28800.543, 28800.393, ..., 28801.242, 28801.055,
         28800.9  ]],

       [[31391.816, 31391.824, 31391.82 , ..., 31391.818, 31391.818,
         31391.812],
        [31390.7  , 31390.688, 31390.686, ..., 31390.674, 31390.674,
         31390.678],
        [31389.584, 31389.588, 31389.611, ..., 31389.533, 31389.56 ,
         31389.572],
...
        [29081.092, 29080.17 , 29079.26 , ..., 29083.926, 29082.973,
         29082.035],
        [29097.678, 29097.049, 29096.48 , ..., 29099.486, 29098.865,
         29098.291],
        [29116.03 , 29115.729, 29115.469, ..., 29116.852, 29116.568,
         29116.303]],

       [[31765.254, 31765.264, 31765.248, ..., 31765.238, 31765.24 ,
         31765.223],
        [31760.299, 31760.291, 31760.316, ..., 31760.314, 31760.305,
         31760.31 ],
        [31754.55 , 31754.543, 31754.562, ..., 31754.463, 31754.486,
         31754.514],
        ...,
        [28666.143, 28665.467, 28664.799, ..., 28668.242, 28667.518,
         28666.81 ],
        [28685.99 , 28685.55 , 28685.117, ..., 28687.338, 28686.88 ,
         28686.424],
        [28708.098, 28707.914, 28707.738, ..., 28708.719, 28708.545,
         28708.35 ]]], shape=(12, 180, 360), dtype=float32)
Coordinates:
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * lat      (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon      (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
Attributes:
    units:      m
    long_name:  10-hPa geopotential height
    product:    ERA5

### 2. Model zonal means and Reanalysis climatologies

In [4]:
%%time
R = sd.load_references(cfg)
for m, ds in R.all.items():
    ok = np.isfinite(ds["U"].sel(plev=REF_LEVEL).isel(lead=0, lat=-1))
    print(f"Reanalysis init{m:02d}: {int(ok.sum())} member-years in {int(ok.any('member').sum())} years")
M = sd.load_model(cfg, E, R)
for field in CLIM_FIELDS:
    for init in INIT_MONTHS:
        c = rc.matched_climatology(field, init, method=CLIM_METHOD, mode=derivation_mode)
    print(f"{field} climatologies ready ({c.attrs.get('climatology_method', '')} ...)")

Reanalysis init05: 300 member-years in 30 years
Reanalysis init11: 300 member-years in 30 years
Reanalysis init02: 60 member-years in 6 years
Reanalysis init08: 60 member-years in 6 years


[init Feb 2026] 20 members, daily 01 Feb 2026 - 31 Jan 2028; climatology: Reanalysis init02 6 starts 1985-2010 (every fifth year)


[init May 2026] 30 members, daily 01 May 2026 - 30 Apr 2028; climatology: Reanalysis init05 1981-2010


[init Aug 2026] 30 members, daily 01 Aug 2026 - 31 Jul 2028; climatology: Reanalysis init08 6 starts 1985-2010 (every fifth year)


[WARN] reader pool failed (BrokenProcessPool); reading the remaining 10 jobs serially


[WARN] reader pool failed (BrokenProcessPool); reading the remaining 10 jobs serially


... (4 outputs left out)

[init Nov 2026] 30 members, daily 01 Nov 2026 - 31 Oct 2028; climatology: Reanalysis init11 1981-2010


Z10 climatologies ready (Reanalysis init11 1981-2010 ...)


PSL climatologies ready (Reanalysis init11 1981-2010 ...)


TS climatologies ready (Reanalysis init11 1981-2010 ...)
CPU times: user 3min 20s, sys: 6min 25s, total: 9min 46s
Wall time: 2h 16min 49s


### 3. Model daily 10-hPa maps

In [5]:
%%time
for init in INIT_MONTHS:
    maps = sc.pooled_daily_maps(SIMULATION_ROOT, MAP_FILE, POOL, YEAR, init, mode=derivation_mode, workers=WORKERS)
    print(f"[init {MONTH_NAMES[init]}] {maps.sizes['member']} members, {maps.sizes['time']} days "
          f"({pd.Timestamp(maps['time'].values[0]):%b %Y} - {pd.Timestamp(maps['time'].values[-1]):%b %Y}, Sep-May)")

/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_s2d_proc/diagnostic/util/strat_case.py:630: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 61. This could degrade performance. Instead, consider rechunking after loading.
  return xr.open_dataset(path, chunks={"time": 61})    # lazy: about 0.3 GB per case and start


/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_s2d_proc/diagnostic/util/strat_case.py:630: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 61. This could degrade performance. Instead, consider rechunking after loading.
  return xr.open_dataset(path, chunks={"time": 61})    # lazy: about 0.3 GB per case and start


[Reanalysis-OBS 2026-02] repeats an already pooled case member for member; not pooled again


/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_s2d_proc/diagnostic/util/strat_case.py:630: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 61. This could degrade performance. Instead, consider rechunking after loading.
  return xr.open_dataset(path, chunks={"time": 61})    # lazy: about 0.3 GB per case and start


[init Feb] 20 members, 546 days (Feb 2026 - Jan 2028, Sep-May)


/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_s2d_proc/diagnostic/util/strat_case.py:630: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 61. This could degrade performance. Instead, consider rechunking after loading.
  return xr.open_dataset(path, chunks={"time": 61})    # lazy: about 0.3 GB per case and start


... (10 outputs left out)

[WARN] reader pool failed (BrokenProcessPool); reading the remaining 10 jobs serially


[RTENSO2026_obs_ice_nsidc 2026-11] reading Z010 maps of 10 members


[WARN] reader pool failed (BrokenProcessPool); reading the remaining 10 jobs serially


[init Nov] 30 members, 546 days (Nov 2026 - Oct 2028, Sep-May)
CPU times: user 48.1 s, sys: 7.35 s, total: 55.4 s
Wall time: 24min 36s


### 4. Cache inventory and the daily-minus-monthly height offset

Every cache with its size. Then the one correction applied to the model data: the native daily `Z`
(eam.h1, computed online on pressure levels) is higher than the monthly `Z3` the climatology is made
from (interpolated afterwards; extrapolated below ground at 1000 hPa); U, T and PSL agree to 0.1. A
processing difference is nearly constant across leads (seasons) and starts, a physical signal is not:
the offset is about 38 m at 10 hPa with a standard deviation of about 2 m over 24 leads, identical in
the three starts, so adding it to the climatology removes a processing artifact, not signal.

In [6]:
root = rc.diag_dir("")
rows = []
for pattern in ("*/strat/*/*.nc", "observations/strat/*.nc", "observations/strat/*/*.nc",
                "Reanalysis/leadtime_clim/*/Z10/*.nc", "Reanalysis/leadtime_clim/*/PSL/*init0[28]*.nc"):
    for f in sorted(root.glob(pattern)):
        rows.append({"cache": str(f.relative_to(root)), "MB": round(f.stat().st_size / 1e6, 1)})
inventory = pd.DataFrame(rows)
inventory.to_csv(DATA_DIR / "strat_cache_inventory.csv", index=False)
print(f"{len(inventory)} caches, {inventory['MB'].sum() / 1e3:.1f} GB under {root}")
display(inventory)
print("Polar-cap (60-90N) daily-minus-monthly Z offset (ensemble mean) by start: mean and std over 24 leads (m)")
display(sd.offset_table(M).round(1))

49 caches, 7.5 GB under /global/cfs/cdirs/e3sm/S2S2D/s2d_diag


,cache,MB
0,RTENSO2026_clim/strat/daily_maps/RTENSO2026_cl...,263.2
1,RTENSO2026_clim/strat/daily_maps/RTENSO2026_cl...,263.2
2,RTENSO2026_clim/strat/daily_maps/RTENSO2026_cl...,264.0
3,RTENSO2026_clim/strat/daily_maps/RTENSO2026_cl...,263.8
4,RTENSO2026_clim/strat/daily_zonal/RTENSO2026_c...,285.9
5,RTENSO2026_clim/strat/daily_zonal/RTENSO2026_c...,285.9
6,RTENSO2026_clim/strat/daily_zonal/RTENSO2026_c...,285.9
7,RTENSO2026_clim/strat/daily_zonal/RTENSO2026_c...,285.9
8,RTENSO2026_clim/strat/monthly_zonal/RTENSO2026...,6.0
9,RTENSO2026_clim/strat/monthly_zonal/RTENSO2026...,6.0


Polar-cap (60-90N) daily-minus-monthly Z offset (ensemble mean) by start: mean and std over 24 leads (m)


mean (m)                      std over leads                 
start           Aug    Feb    May    Nov            Aug   Feb  May   Nov
plev (hPa)                                                              
10.0           38.0   38.1   38.0   38.0            1.9   1.9  1.9   1.9
50.0            0.7    0.7    0.7    0.6            0.0   0.1  0.0   0.0
100.0           5.3    5.3    5.3    5.3            0.1   0.1  0.1   0.1
500.0           2.6    2.6    2.6    2.6            0.3   0.3  0.3   0.3
1000.0        211.9  210.8  213.2  212.2           11.1  11.1  9.7  10.1